In [34]:
import pandas as pd
from pathlib import Path

base_dir = Path.cwd().parent
raw_dir = base_dir / "data" / "raw"
processed_dir = base_dir / "data" / "processed"
recommendations_path = raw_dir / "recommendations.csv"

In [21]:
games = pd.read_csv(processed_dir / "games_cleaned.csv")
users = pd.read_csv(processed_dir / "users_cleaned.csv")

print("games:", games.shape)
print("users:", users.shape)

games: (50872, 20)
users: (14306064, 3)


#### 게임 feature

In [22]:
sample_chunk = pd.read_csv(
    recommendations_path,
    nrows=200_000
)

print(sample_chunk.shape)
print(sample_chunk.columns.tolist())
print(sample_chunk.head())

(200000, 8)
['app_id', 'helpful', 'funny', 'date', 'is_recommended', 'hours', 'user_id', 'review_id']
    app_id  helpful  funny        date  is_recommended  hours  user_id  \
0   975370        0      0  2022-12-12            True   36.3    51580   
1   304390        4      0  2017-02-17           False   11.5     2586   
2  1085660        2      0  2019-11-17            True  336.5   253880   
3   703080        0      0  2022-09-23            True   27.4   259432   
4   526870        0      0  2021-01-10            True    7.9    23869   

   review_id  
0          0  
1          1  
2          2  
3          3  
4          4  


#### 게임별 집계

In [23]:
game_parts = []

for chunk in pd.read_csv(
    recommendations_path,
    chunksize=200_000
):
    chunk["is_recommended"] = chunk["is_recommended"].astype(int)

    part = chunk.groupby("app_id").agg(
        review_count=("review_id", "count"),
        recommended_count=("is_recommended", "sum"),
        hours_sum=("hours", "sum"),
        helpful_sum=("helpful", "sum"),
        funny_sum=("funny", "sum")
    ).reset_index()

    game_parts.append(part)

print("Chunks processed:", len(game_parts))

Chunks processed: 206


In [24]:
game_agg = pd.concat(game_parts, ignore_index=True)

game_agg = game_agg.groupby("app_id").agg(
    review_count=("review_count", "sum"),
    recommended_count=("recommended_count", "sum"),
    hours_sum=("hours_sum", "sum"),
    helpful_sum=("helpful_sum", "sum"),
    funny_sum=("funny_sum", "sum")
).reset_index()

print(game_agg.shape)
print(game_agg.head())

(37610, 6)
   app_id  review_count  recommended_count   hours_sum  helpful_sum  funny_sum
0      10         41043              39332  10087417.3       154158      48541
1      20          4284               3685    150714.4        20586      11237
2      30          4432               3977    347225.9         8874       3823
3      40          1610               1300     48115.6         4715       1292
4      50          9721               9283    179548.9        11384       2801


In [25]:
game_agg["recommend_rate"] = (
    game_agg["recommended_count"] /
    game_agg["review_count"]
)

game_agg["avg_hours"] = (
    game_agg["hours_sum"] /
    game_agg["review_count"]
)

game_agg["avg_helpful"] = (
    game_agg["helpful_sum"] /
    game_agg["review_count"]
)

game_agg["avg_funny"] = (
    game_agg["funny_sum"] /
    game_agg["review_count"]
)

print(
    game_agg[
        [
            "app_id",
            "review_count",
            "recommend_rate",
            "avg_hours",
            "avg_helpful",
            "avg_funny"
        ]
    ].head()
)

   app_id  review_count  recommend_rate   avg_hours  avg_helpful  avg_funny
0      10         41043        0.958312  245.776802     3.756012   1.182686
1      20          4284        0.860177   35.180766     4.805322   2.623016
2      30          4432        0.897338   78.345194     2.002256   0.862590
3      40          1610        0.807453   29.885466     2.928571   0.802484
4      50          9721        0.954943   18.470209     1.171073   0.288139


In [26]:
game_features = games.merge(
    game_agg,
    on="app_id",
    how="left"
)

print(game_features.shape)
print(game_features.columns.tolist())

(50872, 29)
['app_id', 'title', 'date_release', 'win', 'mac', 'linux', 'rating', 'positive_ratio', 'user_reviews', 'price_final', 'price_original', 'discount', 'steam_deck', 'rating_score', 'release_year', 'release_month', 'is_free', 'price_change', 'discount_rate', 'platform_count', 'review_count', 'recommended_count', 'hours_sum', 'helpful_sum', 'funny_sum', 'recommend_rate', 'avg_hours', 'avg_helpful', 'avg_funny']


In [27]:
game_features.to_csv(
    processed_dir / "game_features.csv",
    index=False
)

print("Saved:", processed_dir / "game_features.csv")

Saved: C:\Users\color\Desktop\steam-game-analysis\data\processed\game_features.csv


In [28]:
user_parts = []

for chunk in pd.read_csv(
    recommendations_path,
    chunksize=200_000
):
    chunk["is_recommended"] = chunk["is_recommended"].astype(int)

    part = chunk.groupby("user_id").agg(
        recommendation_count=("review_id", "count"),
        recommended_count=("is_recommended", "sum"),
        hours_sum=("hours", "sum"),
        helpful_sum=("helpful", "sum"),
        funny_sum=("funny", "sum")
    ).reset_index()

    user_parts.append(part)

print("Chunks processed:", len(user_parts))

Chunks processed: 206


In [29]:
user_agg = pd.concat(
    user_parts,
    ignore_index=True
)

user_agg = user_agg.groupby("user_id").agg(
    recommendation_count=("recommendation_count", "sum"),
    recommended_count=("recommended_count", "sum"),
    hours_sum=("hours_sum", "sum"),
    helpful_sum=("helpful_sum", "sum"),
    funny_sum=("funny_sum", "sum")
).reset_index()

print(user_agg.shape)
print(user_agg.head())

(13781059, 6)
   user_id  recommendation_count  recommended_count  hours_sum  helpful_sum  \
0        0                    28                 11     1498.9          290   
1        1                     1                  0       14.9            0   
2        2                     4                  4      170.9            0   
3        3                     2                  2      488.2            0   
4        4                     1                  1        6.1           83   

   funny_sum  
0         44  
1          0  
2          0  
3          0  
4        221  


In [30]:
user_agg["recommend_rate"] = (
    user_agg["recommended_count"] /
    user_agg["recommendation_count"]
)

user_agg["avg_hours"] = (
    user_agg["hours_sum"] /
    user_agg["recommendation_count"]
)

user_agg["avg_helpful"] = (
    user_agg["helpful_sum"] /
    user_agg["recommendation_count"]
)

user_agg["avg_funny"] = (
    user_agg["funny_sum"] /
    user_agg["recommendation_count"]
)

print(
    user_agg[
        [
            "user_id",
            "recommendation_count",
            "recommend_rate",
            "avg_hours",
            "avg_helpful",
            "avg_funny"
        ]
    ].head()
)

   user_id  recommendation_count  recommend_rate   avg_hours  avg_helpful  \
0        0                    28        0.392857   53.532143    10.357143   
1        1                     1        0.000000   14.900000     0.000000   
2        2                     4        1.000000   42.725000     0.000000   
3        3                     2        1.000000  244.100000     0.000000   
4        4                     1        1.000000    6.100000    83.000000   

    avg_funny  
0    1.571429  
1    0.000000  
2    0.000000  
3    0.000000  
4  221.000000  


In [31]:
users["reviews_per_product"] = (
    users["reviews"] /
    users["products"].replace(0, pd.NA)
)

user_features = users.merge(
    user_agg[
        [
            "user_id",
            "recommendation_count",
            "recommend_rate",
            "avg_hours",
            "avg_helpful",
            "avg_funny"
        ]
    ],
    on="user_id",
    how="left"
)

feature_columns = [
    "recommendation_count",
    "recommend_rate",
    "avg_hours",
    "avg_helpful",
    "avg_funny"
]

user_features[feature_columns] = (
    user_features[feature_columns].fillna(0)
)

print(user_features.shape)
print(user_features.columns.tolist())

(14306064, 9)
['user_id', 'products', 'reviews', 'reviews_per_product', 'recommendation_count', 'recommend_rate', 'avg_hours', 'avg_helpful', 'avg_funny']


In [32]:
user_features.to_csv(
    processed_dir / "user_features.csv",
    index=False
)

print("Saved:", processed_dir / "user_features.csv")

Saved: C:\Users\color\Desktop\steam-game-analysis\data\processed\user_features.csv


In [35]:
print("[Game Features]")
print(game_features.shape)
print(game_features.columns.tolist())

[Game Features]
(50872, 29)
['app_id', 'title', 'date_release', 'win', 'mac', 'linux', 'rating', 'positive_ratio', 'user_reviews', 'price_final', 'price_original', 'discount', 'steam_deck', 'rating_score', 'release_year', 'release_month', 'is_free', 'price_change', 'discount_rate', 'platform_count', 'review_count', 'recommended_count', 'hours_sum', 'helpful_sum', 'funny_sum', 'recommend_rate', 'avg_hours', 'avg_helpful', 'avg_funny']


In [36]:
print("\n[User Features]")
print(user_features.shape)
print(user_features.columns.tolist())


[User Features]
(14306064, 9)
['user_id', 'products', 'reviews', 'reviews_per_product', 'recommendation_count', 'recommend_rate', 'avg_hours', 'avg_helpful', 'avg_funny']
